# Chapter 8: Reasoning, Memory, and Control

**What you'll learn:** How to define reasoning, memory, and control in geometric terms, and run the four-condition control experiment.

**Key concept:** Reasoning = curvature + stretching + dependency (all three needed). Metric control dominates rotation control.

**Time:** ~25 minutes

## 8.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="cuda")

## 8.2 Defining Reasoning Geometrically

A layer is **reasoning** when three conditions hold simultaneously:

1. **Interaction (curvature):** Ω(l,t) is above baseline — genuinely nonlinear processing.
2. **Propagation (stretching):** κ(l) is high — the layer is selectively amplifying directions.
3. **Dependency:** D_l is high — the processing actually matters for the output.

All three must be present. Interaction without dependency is "busy work." Dependency without interaction is just passing information through.

In [ ]:
result = ltg.analyse(
    "If all mammals are warm-blooded and whales are mammals, are whales warm-blooded?",
    model=model
)

# Identify reasoning layers using the three-condition test
curv = result.curvature_by_layer
kappa = result.condition_numbers
dep = result.dependency_profile

# Normalise each to [0, 1] relative to its own range
curv_score = (curv - curv.min()) / (curv.max() - curv.min() + 1e-8)
kappa_score = (kappa - kappa.min()) / (kappa.max() - kappa.min() + 1e-8)
dep_score = (dep - dep.min()) / (dep.max() - dep.min() + 1e-8)

# A layer is "reasoning" if all three scores are above 0.5
n_layers = min(len(curv_score), len(kappa_score), len(dep_score))
reasoning_layers = []
for l in range(n_layers):
    is_reasoning = curv_score[l] > 0.5 and kappa_score[l] > 0.5 and dep_score[l] > 0.5
    if is_reasoning:
        reasoning_layers.append(l)

print(f"Reasoning layers: {reasoning_layers}")
print(f"(out of {n_layers} total layers)")

In [ ]:
# Visualise the three conditions
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Curvature
axes[0].plot(curv_score[:n_layers], color='#EE6677', linewidth=2)
axes[0].axhline(0.5, color='grey', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Curvature\n(normalised)')
axes[0].set_title('Three-Condition Reasoning Test', fontsize=14)
for l in reasoning_layers:
    axes[0].axvspan(l-0.5, l+0.5, alpha=0.2, color='green')

# Condition number
axes[1].plot(kappa_score[:n_layers], color='#AA3377', linewidth=2)
axes[1].axhline(0.5, color='grey', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Condition number\n(normalised)')
for l in reasoning_layers:
    axes[1].axvspan(l-0.5, l+0.5, alpha=0.2, color='green')

# Dependency
axes[2].plot(dep_score[:n_layers], color='#4477AA', linewidth=2)
axes[2].axhline(0.5, color='grey', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Dependency\n(normalised)')
axes[2].set_xlabel('Layer')
for l in reasoning_layers:
    axes[2].axvspan(l-0.5, l+0.5, alpha=0.2, color='green')

plt.tight_layout()
plt.show()
print(f"Green bands = layers where ALL THREE conditions are met = reasoning layers")

## 8.3 Memory: Does Early Information Persist?

**Functional memory** = information from early tokens persists and influences the output in late layers. We measure this through the dependency map.

In [ ]:
text = "Paris is the capital of France. What country is Paris in?"

# Get the full (L, T) dependency map
H_raw = core.extract_hidden_states(model.hf_model, model.tokenizer, text, model.device)
H_np = H_raw[1:].cpu().numpy()
L, T, p = H_np.shape
H_flat = H_np.reshape(L * T, p)
metric = core.estimate_metric(H_flat, n_components=256)
dep_result = core.compute_dependency_density(
    model.hf_model, model.tokenizer, text, metric, device=model.device
)

D_lt = dep_result.D_lt  # shape (L, T)
L, T = D_lt.shape

# Get token labels
tokens = [model.tokenizer.decode([tid]) for tid in model.tokenizer.encode(text)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full dependency map
im = ax1.imshow(D_lt, aspect='auto', cmap='YlOrRd', origin='lower')
ax1.set_xlabel('Token position')
ax1.set_ylabel('Layer')
ax1.set_title('Full Dependency Map D(l,t)')
if T <= 20:
    ax1.set_xticks(range(T))
    ax1.set_xticklabels(tokens[:T], rotation=45, ha='right', fontsize=7)
plt.colorbar(im, ax=ax1)

# Persistence: dependency of first few tokens across layers
for t in [0, 1, 2, T-1]:
    label = tokens[t].strip() if t < len(tokens) else f"t={t}"
    ax2.plot(D_lt[:, t], linewidth=2, label=f'Token {t}: "{label}"')

ax2.set_xlabel('Layer')
ax2.set_ylabel('D(l, t)')
ax2.set_title('Token Dependency Across Layers')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Persistence ratio for first token
persistence = D_lt[-3:, 0].mean() / (D_lt[:3, 0].mean() + 1e-12)
print(f"First token persistence (late/early): {persistence:.3f}")
print(f"  > 0.1 suggests functional memory of the first token")

## 8.4 Control: Metric vs Rotation

We can intervene on the model's computation by modifying either:
- **Metric control:** clamp the eigenvalues of P (change what gets amplified)
- **Rotation control:** perturb U (change the direction of information flow)

The theory predicted dual > rotation > metric > baseline.
The experiments showed: **metric ≈ dual >> rotation > baseline**.

In [ ]:
# Run the four-condition control experiment
ctrl_results = ltg.control_experiment(
    "Explain why ice floats on water",
    model=model,
    metric_strength=0.5,
    rotation_strength=0.3,
)

# Print results
print(f"{'Condition':15s} {'D_total':>10s} {'H_90':>6s} {'Entropy':>10s}")
print("-" * 45)
for name, r in ctrl_results.items():
    print(f"{name:15s} {r.dep_total:10.4f} {r.dep_horizon_90:6d} {r.dep_entropy:10.3f}")

# Visualise
ltg.plot_control(ctrl_results, save_path="ch8_control.png")
img = plt.imread("ch8_control.png")
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(img); ax.axis('off')
plt.show()

In [ ]:
# Compare dependency profiles for all conditions
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'baseline': '#4477AA', 'rotation_only': '#66CCEE', 
          'metric_only': '#EE6677', 'dual': '#AA3377'}

for name, r in ctrl_results.items():
    D_norm = r.dep_profile / r.dep_profile.sum()
    ax.plot(D_norm, linewidth=2, color=colors.get(name, 'grey'), 
            label=f'{name} (S={r.dep_entropy:.2f})')

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Normalised dependency', fontsize=12)
ax.set_title('Dependency Profiles Under Different Control Conditions', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print("\nKey insight: metric-only and dual produce nearly identical profiles.")
print("Rotation control barely changes the profile.")
print("Reason: eigenvalues of P control gradient flow; rotation preserves norms.")

## 8.5 Metric Strength Sweep

How does the strength of metric control affect dependency?

In [ ]:
strengths = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
dep_totals_sweep = []

for s in strengths:
    ctrl = ltg.control_experiment(
        "Explain gravity in simple terms",
        model=model,
        metric_strength=s,
        rotation_strength=0.3,
    )
    if 'metric_only' in ctrl:
        dep_totals_sweep.append(ctrl['metric_only'].dep_total)
    else:
        dep_totals_sweep.append(np.nan)
    print(f"  strength={s:.1f}: D_total={dep_totals_sweep[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strengths, dep_totals_sweep, 'o-', color='#EE6677', linewidth=2, markersize=8)
ax.set_xlabel('Metric control strength', fontsize=12)
ax.set_ylabel('D_total (metric-only)', fontsize=12)
ax.set_title('Effect of Metric Control Strength on Dependency', fontsize=14)
plt.tight_layout()
plt.show()

## 8.6 Key Takeaways

1. **Reasoning = curvature + stretching + dependency** — all three conditions must hold.
2. **Functional memory** = early-token dependency persisting to late layers.
3. **Metric control dominates** — modifying eigenvalues of P is sufficient.
4. **Rotation control adds nothing measurable** — a surprise that corrected the theory.
5. Science works: predict, test, update.

## Exercise

Apply the three-condition reasoning test to five different prompts. Count the number of reasoning layers in each. Do reasoning-heavy prompts (logic puzzles) have more reasoning layers than factual retrieval prompts?

In [ ]:
# Your turn!
# Test prompts of different types and count reasoning layers